In [1]:
import pandas as pd

df = pd.read_csv("../Data/cleaned/hospital_general_info_cleaned.csv")

print("Shape:", df.shape)
print("\nColumns:")
for col in df.columns:
    print(f"  {col}")

Shape: (5426, 32)

Columns:
  facility_id
  facility_name
  address
  citytown
  state
  zip_code
  countyparish
  telephone_number
  hospital_type
  hospital_ownership
  emergency_services
  meets_criteria_for_birthing_friendly_designation
  hospital_overall_rating
  mort_group_measure_count
  count_of_facility_mort_measures
  count_of_mort_measures_better
  count_of_mort_measures_no_different
  count_of_mort_measures_worse
  safety_group_measure_count
  count_of_facility_safety_measures
  count_of_safety_measures_better
  count_of_safety_measures_no_different
  count_of_safety_measures_worse
  readm_group_measure_count
  count_of_facility_readm_measures
  count_of_readm_measures_better
  count_of_readm_measures_no_different
  count_of_readm_measures_worse
  pt_exp_group_measure_count
  count_of_facility_pt_exp_measures
  te_group_measure_count
  count_of_facility_te_measures


In [2]:
cat_cols = [
    "hospital_type", 
    "hospital_ownership", 
    "emergency_services",
    "meets_criteria_for_birthing_friendly_designation",
    "hospital_overall_rating"
]

for col in cat_cols:
    print(f"\n{col}:")
    print(df[col].value_counts(dropna=False).to_string())


hospital_type:
hospital_type
Acute Care Hospitals                    3116
Critical Access Hospitals               1376
Psychiatric                              633
Acute Care - Veterans Administration     132
Childrens                                 94
Rural Emergency Hospital                  39
Acute Care - Department of Defense        32
Long-term                                  4

hospital_ownership:
hospital_ownership
Voluntary non-profit - Private                 2304
Proprietary                                    1069
Government - Hospital District or Authority     519
Government - Local                              401
Voluntary non-profit - Other                    355
Voluntary non-profit - Church                   271
Government - State                              209
Veterans Health Administration                  132
Physician                                        76
Government - Federal                             43
Department of Defense                            3

In [3]:
df["hospital_overall_rating"] = df["hospital_overall_rating"].replace("Not Available", None)
df["hospital_overall_rating"] = pd.to_numeric(df["hospital_overall_rating"])

df = df.rename(columns={
    "meets_criteria_for_birthing_friendly_designation": "birthing_friendly"
})
df["birthing_friendly"] = df["birthing_friendly"].map({"Y": True}).fillna(False)

print("hospital_overall_rating:")
print(df["hospital_overall_rating"].value_counts(dropna=False))
print("\nbirthing_friendly:")
print(df["birthing_friendly"].value_counts(dropna=False))

hospital_overall_rating:
hospital_overall_rating
NaN    2560
3.0     935
4.0     765
2.0     649
5.0     288
1.0     229
Name: count, dtype: int64

birthing_friendly:
birthing_friendly
False    3161
True     2265
Name: count, dtype: int64


C:\Users\Fatima zahra\AppData\Local\Temp\ipykernel_8896\2983731324.py:7: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df["birthing_friendly"] = df["birthing_friendly"].map({"Y": True}).fillna(False)


In [4]:
numeric_cols = [col for col in df.columns if "count" in col.lower() or "rating" in col.lower()]

for col in numeric_cols:
    non_numeric = df[col].dropna().apply(
        lambda x: not str(x).replace(".", "").replace("-", "").isnumeric()
    ).sum()
    print(f"{col}: {non_numeric} non-numeric values")

countyparish: 5426 non-numeric values
hospital_overall_rating: 0 non-numeric values
mort_group_measure_count: 875 non-numeric values
count_of_facility_mort_measures: 1786 non-numeric values
count_of_mort_measures_better: 1786 non-numeric values
count_of_mort_measures_no_different: 1786 non-numeric values
count_of_mort_measures_worse: 1786 non-numeric values
safety_group_measure_count: 875 non-numeric values
count_of_facility_safety_measures: 2073 non-numeric values
count_of_safety_measures_better: 2073 non-numeric values
count_of_safety_measures_no_different: 2073 non-numeric values
count_of_safety_measures_worse: 2073 non-numeric values
readm_group_measure_count: 875 non-numeric values
count_of_facility_readm_measures: 1160 non-numeric values
count_of_readm_measures_better: 1160 non-numeric values
count_of_readm_measures_no_different: 1160 non-numeric values
count_of_readm_measures_worse: 1160 non-numeric values
pt_exp_group_measure_count: 875 non-numeric values
count_of_facility_pt_e

In [5]:
for col in numeric_cols:
    if col == "countyparish":
        continue  # skip, it's actually a text column
    unique_bad = df[col].dropna().apply(
        lambda x: str(x) if not str(x).replace(".", "").replace("-", "").isnumeric() else None
    ).dropna().unique()
    if len(unique_bad) > 0:
        print(f"{col}: {unique_bad}")

mort_group_measure_count: ['Not Available']
count_of_facility_mort_measures: ['Not Available']
count_of_mort_measures_better: ['Not Available']
count_of_mort_measures_no_different: ['Not Available']
count_of_mort_measures_worse: ['Not Available']
safety_group_measure_count: ['Not Available']
count_of_facility_safety_measures: ['Not Available']
count_of_safety_measures_better: ['Not Available']
count_of_safety_measures_no_different: ['Not Available']
count_of_safety_measures_worse: ['Not Available']
readm_group_measure_count: ['Not Available']
count_of_facility_readm_measures: ['Not Available']
count_of_readm_measures_better: ['Not Available']
count_of_readm_measures_no_different: ['Not Available']
count_of_readm_measures_worse: ['Not Available']
pt_exp_group_measure_count: ['Not Available']
count_of_facility_pt_exp_measures: ['Not Available']
te_group_measure_count: ['Not Available']
count_of_facility_te_measures: ['Not Available']


In [6]:
cols_to_fix = [col for col in numeric_cols if col != "countyparish"]

for col in cols_to_fix:
    df[col] = df[col].replace("Not Available", None)
    df[col] = pd.to_numeric(df[col])

# verify
print("data types after fix:")
print(df[cols_to_fix].dtypes)
print("\nnull counts:")
print(df[cols_to_fix].isnull().sum())

data types after fix:
hospital_overall_rating                  float64
mort_group_measure_count                 float64
count_of_facility_mort_measures          float64
count_of_mort_measures_better            float64
count_of_mort_measures_no_different      float64
count_of_mort_measures_worse             float64
safety_group_measure_count               float64
count_of_facility_safety_measures        float64
count_of_safety_measures_better          float64
count_of_safety_measures_no_different    float64
count_of_safety_measures_worse           float64
readm_group_measure_count                float64
count_of_facility_readm_measures         float64
count_of_readm_measures_better           float64
count_of_readm_measures_no_different     float64
count_of_readm_measures_worse            float64
pt_exp_group_measure_count               float64
count_of_facility_pt_exp_measures        float64
te_group_measure_count                   float64
count_of_facility_te_measures            float6

In [7]:
print("duplicate facility_ids:", df["facility_id"].duplicated().sum())
print("total rows:", len(df))

# final shape
print(f"\nfinal shape: {df.shape}")
print(f"\ncolumn dtypes summary:")
print(df.dtypes.value_counts())

duplicate facility_ids: 0
total rows: 5426

final shape: (5426, 32)

column dtypes summary:
float64    20
object     10
int64       1
bool        1
Name: count, dtype: int64


In [8]:
df.to_csv("../Data/cleaned/hospital_general_info_cleaned.csv", index=False)
print("hospital_general_info saved")

hospital_general_info saved
